# RecursiveChunker Demo
Shows recursive delimiter splitting on a flat document and tree-walk chunking on a tree document.

## Imports

In [ ]:
# Standard Library
from pathlib import Path

# Third Party Library

# Private Library
from cleave.chunker.recursive import RecursiveChunker
from cleave.parsers.factory import ParserFactory
from cleave.schemas import ChunkParams, ContentBlock, ContentType, Document, DocumentPage, Source, SourceType

## Fixture

In [2]:
from tests.fixtures.md import create_sample_md

FIXTURE_PATH = Path("tests/fixtures/sample.md")
if not FIXTURE_PATH.exists():
    create_sample_md(FIXTURE_PATH)

print(f"Fixture: {FIXTURE_PATH}")
print(FIXTURE_PATH.read_text(encoding="utf-8"))

Fixture: tests\fixtures\sample.md
# Sample Document

## Introduction

This is the introduction paragraph.

## Data Overview

This section covers data.

![Sample Image](sample_image.png)

| Name | Value |
| --- | --- |
| Alpha | 1 |
| Beta | 2 |

## Conclusion

Final remarks go here.



In [3]:
flat_doc_parser = ParserFactory.create(str(FIXTURE_PATH), mode="flat")
flat_doc = flat_doc_parser.parse()

print(f"Flat pages       : {len(flat_doc.pages)}")

Flat pages       : 1


In [4]:
tree_doc_parser = ParserFactory.create(str(FIXTURE_PATH), mode="tree")
tree_doc = tree_doc_parser.parse()

print(f"Tree root children : {len(tree_doc.root.children)}")

Tree root children : 1


## Flat document - delimiter splitting

In [5]:
CHUNK_SIZE = 200
CHUNK_OVERLAP = 30

chunker = RecursiveChunker(ChunkParams(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP))
flat_chunks = chunker.chunk(flat_doc)

print(f"chunk_size={CHUNK_SIZE}  chunk_overlap={CHUNK_OVERLAP}")
print(f"Total chunks : {len(flat_chunks)}\n")
for c in flat_chunks:
    print(f"[{c.index}] chars {c.char_start:>4}\u2013{c.char_end:<4}  tokens={c.token_count:>3}  \u2502 {c.text[:70]!r}")

chunk_size=200  chunk_overlap=30
Total chunks : 1

[0] chars    0–149   tokens= 30  │ '# Sample Document\n## Introduction\nThis is the introduction paragraph.\n'


## Tree document - node-walk chunking

In [6]:
tree_chunks = chunker.chunk(tree_doc)

print(f"Total chunks from tree : {len(tree_chunks)}\n")
for c in tree_chunks:
    print(f"[{c.index}] chars {c.char_start:>4}\u2013{c.char_end:<4}  tokens={c.token_count:>3}  \u2502 {c.text[:70]!r}")

Total chunks from tree : 5

[0] chars    0–48    tokens=  8  │ 'Introduction\nThis is the introduction paragraph.'
[1] chars   48–73    tokens=  5  │ 'This section covers data.'
[2] chars   73–207   tokens=  0  │ 'data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAABQAAAAUCAIAAAAC64paAAAA'
[3] chars  207–264   tokens= 22  │ '| Name | Value |\n| --- | --- |\n| Alpha | 1 |\n| Beta | 2 |'
[4] chars  264–297   tokens=  7  │ 'Conclusion\nFinal remarks go here.'


## Table Atomicity


Tables are never split — the table node should appear as a single chunk regardless of size.

In [7]:
table_chunks = [c for c in tree_chunks if "|" in c.text]

print(f"Table chunks : {len(table_chunks)}  (expected 1)")
if table_chunks:
    print(table_chunks[0].text)

Table chunks : 1  (expected 1)
| Name | Value |
| --- | --- |
| Alpha | 1 |
| Beta | 2 |


## Bridge pattern - tree \u2192 flat \u2192 chunk


Serialise a tree document back to Markdown, wrap it in a flat `Document`, then chunk as normal.

In [8]:
md_string = tree_doc.to_markdown()
print("Markdown from tree:\n")
print(md_string)

Markdown from tree:

# Sample Document

## Introduction

This is the introduction paragraph.

## Data Overview

This section covers data.

![Sample Image](sample_image.png)

| Name | Value |
| --- | --- |
| Alpha | 1 |
| Beta | 2 |

## Conclusion

Final remarks go here.


In [10]:
bridge_source = Source(source_type=SourceType.markdown, name="bridge", location="bridge")
bridge_page = DocumentPage(
    page_number=None,
    blocks=[ContentBlock(type=ContentType.text, content=md_string, position=0)],
)
bridge_doc = Document(source=bridge_source, pages=[bridge_page], total_pages=1)

bridge_chunks = chunker.chunk(bridge_doc)

print(f"Bridge chunks : {len(bridge_chunks)}")
for c in bridge_chunks:
    print(f"[{c.index}] {c.text[:80]!r}")

Bridge chunks : 2
[0] '# Sample Document\n\n## Introduction\n\nThis is the introduction paragraph.'
[1] '## Data Overview\n\nThis section covers data.\n\n![Sample Image](sample_image.png)\n\n'


## Custom Delimiters


Pass `custom_delimiters` to override the default hierarchy. Use `DELIMITER_PRESETS` for common languages.

In [13]:
print("Available presets:", list(RecursiveChunker.DELIMITER_PRESETS.keys()))

PYTHON_CODE = """\
class DataLoader:
    def __init__(self, path: str):
        self.path = path

    def load(self):
        with open(self.path) as f:
            return f.read()

    def load_lines(self):
        with open(self.path) as f:
            return f.readlines()


class Tokenizer:
    def __init__(self, vocab_size: int = 1000):
        self.vocab_size = vocab_size
        self.vocab = {}

    def fit(self, corpus: list[str]):
        words = sorted({w for doc in corpus for w in doc.split()})
        self.vocab = {w: i for i, w in enumerate(words[: self.vocab_size])}

    def encode(self, text: str) -> list[int]:
        return [self.vocab[w] for w in text.split() if w in self.vocab]

    def decode(self, ids: list[int]) -> str:
        inv = {v: k for k, v in self.vocab.items()}
        return " ".join(inv[i] for i in ids if i in inv)
"""

python_chunker = RecursiveChunker(
    ChunkParams(chunk_size=200, chunk_overlap=0),
    custom_delimiters=RecursiveChunker.DELIMITER_PRESETS["python"],
)

python_source = Source(source_type=SourceType.txt, name="snippet.py", location="snippet.py")
python_page = DocumentPage(
    page_number=None,
    blocks=[ContentBlock(type=ContentType.text, content=PYTHON_CODE, position=0)],
)
python_doc = Document(source=python_source, pages=[python_page], total_pages=1)
python_chunks = python_chunker.chunk(python_doc)

print(f"\nPython preset — {len(python_chunks)} chunks:\n")
for c in python_chunks:
    print(f"[{c.index}] chars={len(c.text):>3}  tokens={c.token_count:>3}")
    print(c.text)
    print()

Available presets: ['default', 'python', 'js', 'markdown', 'html', 'latex']

Python preset — 6 chunks:

[0] chars=161  tokens= 39
class DataLoader:
    def __init__(self, path: str):
        self.path = path

    def load(self):
        with open(self.path) as f:
            return f.read()

[1] chars= 89  tokens= 19
def load_lines(self):
        with open(self.path) as f:
            return f.readlines()

[2] chars=125  tokens= 33
class Tokenizer:
    def __init__(self, vocab_size: int = 1000):
        self.vocab_size = vocab_size
        self.vocab = {}

[3] chars=176  tokens= 46
def fit(self, corpus: list[str]):
        words = sorted({w for doc in corpus for w in doc.split()})
        self.vocab = {w: i for i, w in enumerate(words[: self.vocab_size])}

[4] chars=113  tokens= 31
def encode(self, text: str) -> list[int]:
        return [self.vocab[w] for w in text.split() if w in self.vocab]

[5] chars=149  tokens= 45
def decode(self, ids: list[int]) -> str:
        inv = {v: k for k

## Multiple File Types


`RecursiveChunker` is format-agnostic - it operates on `Document` objects regardless of the original file type. Here we run the same chunker across all four fixture formats.

In [14]:
from tests.fixtures.txt import create_sample_txt
from tests.fixtures.pdf import create_sample_pdf
from tests.fixtures.docx import create_sample_docx

fixtures = [
    Path("tests/fixtures/sample.md"),
    Path("tests/fixtures/sample.txt"),
    Path("tests/fixtures/sample.pdf"),
    Path("tests/fixtures/sample.docx"),
]

creators = {
    ".md":   (lambda p: __import__("tests.fixtures.md",   fromlist=["create_sample_md"]).create_sample_md(p)),
    ".txt":  (lambda p: create_sample_txt(p)),
    ".pdf":  (lambda p: create_sample_pdf(p)),
    ".docx": (lambda p: create_sample_docx(p)),
}

for fp in fixtures:
    if not fp.exists():
        creators[fp.suffix](fp)

fmt_chunker = RecursiveChunker(ChunkParams(chunk_size=200, chunk_overlap=30))

print(f"{'File':<20} {'Pages':>5}  {'Chunks':>6}  {'Avg tokens':>10}")
print("-" * 48)

for fp in fixtures:
    doc = ParserFactory.create(str(fp), mode="flat").parse()
    chunks = fmt_chunker.chunk(doc)
    avg_tok = sum(c.token_count for c in chunks) / len(chunks) if chunks else 0
    print(f"{fp.name:<20} {len(doc.pages):>5}  {len(chunks):>6}  {avg_tok:>10.1f}")

File                 Pages  Chunks  Avg tokens
------------------------------------------------
sample.md                1       1        30.0
sample.txt               1       1        44.0
sample.pdf               1       1        38.0
sample.docx              1       1        26.0
